# Sección 3 — Arquitecturas de los 5 Modelos
**Entradas:** `artifacts/preprocessing_config.json`  
**Salidas:** `artifacts/models_ready.flag` (marca que las clases fueron validadas)  

Este notebook define, instancia y verifica el forward pass de las 5 arquitecturas. El entrenamiento efectivo ocurre en el Notebook 4.

# Validación Teórica de Arquitecturas y Consistencia Dimensional

## Propósito de este notebook

Este cuaderno **no entrena ningún modelo**. Su función es estrictamente la de un
*dry run* o prueba en vacío: instanciar cada una de las cinco arquitecturas y
verificar que el flujo tensorial de entrada a salida es dimensionalmente consistente
antes de invertir tiempo de GPU en el entrenamiento.

Esta práctica es estándar en el desarrollo de modelos de Deep Learning porque los
errores de dimensión en PyTorch generalmente se manifiestan como excepciones en
tiempo de ejecución durante el primer forward pass, no en tiempo de compilación.
Detectarlos aquí, con batches sintéticos de tamaño controlado, es considerablemente
más eficiente que hacerlo tras horas de entrenamiento.

## ¿Qué se verifica por arquitectura?

- **BiLSTM bidireccional:** se comprueba que la concatenación de los estados ocultos
  forward y backward de la última capa produce un tensor de forma
  `(batch_size, lstm_units × 2)`, y que la capa lineal final lo proyecta
  correctamente a `(batch_size, num_classes)`.

- **CNN-1D multi-kernel:** se verifica que cada rama `Conv2d` con kernel
  `(k, embedding_dim)` reduce la dimensión de secuencia correctamente tras el
  max-pooling, y que la concatenación de las tres ramas produce un vector de
  `(batch_size, num_filters × 3)` sin pérdida de información.

- **TransformerClassifier (DistilBERT):** se comprueba que el tokenizador genera
  tensores con la dimensión `max_length` correcta y que la cabeza de clasificación
  reemplazada produce logits de forma `(batch_size, num_classes)`.

- **FastText Mean:** se verifica que la máscara de padding se aplica correctamente
  antes del promedio, para que los tokens `<PAD>` (índice 0) no desplacen la media
  del embedding hacia el vector cero.

- **TF-IDF + XGBoost:** se valida la instanciación del vectorizador con los parámetros
  de `ngram_range` y `max_features` sin necesidad de datos reales.

## Integridad del pipeline

Al final del notebook se genera el archivo **`artifacts/models_ready.flag`**. El
Notebook 4 comprueba la existencia de este archivo antes de iniciar cualquier
entrenamiento, garantizando que el pipeline no puede avanzar si la validación
dimensional no se completó exitosamente. Este mecanismo evita errores silenciosos
por ejecución desordenada de los notebooks.

In [ ]:
import json, pickle
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import numpy as np
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["CUDA_MODULE_LOADING"] = "LAZY"

from transformers import AutoModelForSequenceClassification, AutoTokenizer

Path("checkpoints").mkdir(exist_ok=True)

with open("artifacts/preprocessing_config.json") as f:
    cfg = json.load(f)

VOCAB_SIZE      = cfg["vocab_size"]
NUM_CLASSES     = cfg["num_classes"]
SEQUENCE_LENGTH = cfg["sequence_length"]
EMBEDDING_DIM   = cfg["embedding_dim"]
CLASS_NAMES     = cfg["class_names"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device        : {DEVICE}")
print(f"vocab_size    : {VOCAB_SIZE}")
print(f"num_classes   : {NUM_CLASSES}")
print(f"seq_length    : {SEQUENCE_LENGTH}")
print(f"embedding_dim : {EMBEDDING_DIM}")

## Modelo 1 — DistilBERT / RoBERTa (Transfer Learning)

In [ ]:
class TransformerClassifier:
    """
    Wrapper sobre AutoModelForSequenceClassification de HuggingFace.
    Estandariza la interfaz train_mode / eval_mode / forward para que el
    bucle de entrenamiento del Notebook 4 sea agnóstico al backbone.

    El checkpoint se selecciona por clave corta para facilitar el cambio
    entre DistilBERT y RoBERTa sin tocar el código de entrenamiento.
    """
    CHECKPOINTS = {
        "distilbert": "distilbert-base-uncased",
        "roberta":    "roberta-base",
    }

    def __init__(
        self,
        num_classes: int,
        model_key: str = "distilbert",
        max_length: int = 128,
        device: Optional[torch.device] = None,
    ):
        self.num_classes = num_classes
        self.max_length  = max_length
        self.device      = device or torch.device("cpu")
        checkpoint       = self.CHECKPOINTS.get(model_key, model_key)

        print(f"INFO | Cargando checkpoint: {checkpoint}")
        self.tokenizer = AutoTokenizer.from_pretrained(checkpoint)
        self.model     = AutoModelForSequenceClassification.from_pretrained(
            checkpoint,
            num_labels=num_classes,
            ignore_mismatched_sizes=True,   # reemplaza la cabeza de clasificación
        ).to(self.device)

    def tokenize(self, texts: list) -> dict:
        return self.tokenizer(
            texts,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

    def parameters(self):
        return self.model.parameters()

    def train_mode(self):  self.model.train()
    def eval_mode(self):   self.model.eval()

    def forward(self, batch: dict) -> torch.Tensor:
        batch   = {k: v.to(self.device) for k, v in batch.items()}
        outputs = self.model(**batch)
        return outputs.logits


# Verificación: forward pass con batch sintético
print("\n--- Test TransformerClassifier ---")
tf_clf = TransformerClassifier(
    num_classes=NUM_CLASSES,
    model_key="distilbert",
    device=DEVICE,
)
dummy_enc = tf_clf.tokenize(["test utterance", "another sample"])
logits = tf_clf.forward(dummy_enc)
print(f"  Logits shape: {logits.shape}  (esperado: [2, {NUM_CLASSES}])")

## Modelo 2 — BiLSTM bidireccional + Word2Vec
Todos los hiperparámetros están expuestos en `BiLSTMConfig` para facilitar grid search.

In [ ]:
@dataclass
class BiLSTMConfig:
    """
    Dataclass de configuración para la BiLSTM.
    Rangos del grid search (del enunciado):
      lstm_units      : [32, 64, 128]
      num_lstm_layers : [1, 2]
      dropout_rate    : [0.3, 0.5]
      learning_rate   : [0.0005, 0.001, 0.003]
      batch_size      : [32, 64]
      sequence_length : [50, 100]
      embedding_dim   : [100, 200]
      epochs          : [20, 50]
    """
    vocab_size:       int
    num_classes:      int
    embedding_dim:    int   = 100
    lstm_units:       int   = 128
    num_lstm_layers:  int   = 2
    dropout_rate:     float = 0.3
    sequence_length:  int   = 100
    learning_rate:    float = 0.001
    batch_size:       int   = 64
    epochs:           int   = 20
    freeze_embeddings: bool = False
    pretrained_weights: Optional[np.ndarray] = field(default=None, repr=False)


class BiLSTMClassifier(nn.Module):
    """
    LSTM bidireccional apilada con dropout variacional entre capas.

    Representación de sentencia: concatenación del estado oculto forward
    y backward de la última capa — captura el contexto global sin pérdida
    de información que produciría un mean-pooling en secuencias cortas.
    """
    def __init__(self, cfg: BiLSTMConfig):
        super().__init__()
        self.cfg = cfg

        if cfg.pretrained_weights is not None:
            weights = torch.tensor(cfg.pretrained_weights, dtype=torch.float32)
            self.embedding = nn.Embedding.from_pretrained(
                weights, freeze=cfg.freeze_embeddings, padding_idx=0
            )
        else:
            self.embedding = nn.Embedding(
                cfg.vocab_size, cfg.embedding_dim, padding_idx=0
            )

        # PyTorch solo aplica dropout entre capas cuando num_layers > 1
        self.lstm = nn.LSTM(
            input_size=cfg.embedding_dim,
            hidden_size=cfg.lstm_units,
            num_layers=cfg.num_lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=cfg.dropout_rate if cfg.num_lstm_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(cfg.dropout_rate)
        self.fc      = nn.Linear(cfg.lstm_units * 2, cfg.num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len)
        emb = self.dropout(self.embedding(x))       # (batch, seq, emb_dim)
        _, (hidden, _) = self.lstm(emb)
        # hidden: (num_layers * 2, batch, lstm_units)
        fwd = hidden[-2]   # última capa, dirección forward
        bwd = hidden[-1]   # última capa, dirección backward
        rep = self.dropout(torch.cat([fwd, bwd], dim=1))  # (batch, lstm*2)
        return self.fc(rep)


# Verificación con embeddings pre-entrenados
w2v_matrix = np.load("artifacts/w2v_matrix.npy")
bilstm_cfg = BiLSTMConfig(
    vocab_size=VOCAB_SIZE, num_classes=NUM_CLASSES,
    embedding_dim=EMBEDDING_DIM, sequence_length=SEQUENCE_LENGTH,
    pretrained_weights=w2v_matrix,
)
bilstm = BiLSTMClassifier(bilstm_cfg).to(DEVICE)
dummy_x = torch.randint(0, VOCAB_SIZE, (4, SEQUENCE_LENGTH)).to(DEVICE)
print(f"\n--- Test BiLSTM ---")
print(f"  Logits shape: {bilstm(dummy_x).shape}  (esperado: [4, {NUM_CLASSES}])")
print(f"  Parámetros  : {sum(p.numel() for p in bilstm.parameters()):,}")

## Modelo 3 — CNN-1D (Multi-kernel)

In [ ]:
class CNN1DClassifier(nn.Module):
    """
    Arquitectura Kim (2014) con tres ramas convolucionales paralelas de
    ancho de kernel 3, 4 y 5. Cada rama extrae n-gramas locales de diferente
    longitud y max-over-time pooling reduce cada mapa de características
    a un escalar — la representación final es independiente de seq_len.

    Se usa Conv2d (en lugar de Conv1d) porque el tratamiento de la matriz
    de embedding como imagen 2D (H=seq_len, W=emb_dim) produce gradientes
    más estables durante el fine-tuning de embeddings.
    """
    def __init__(
        self,
        vocab_size:    int,
        num_classes:   int,
        embedding_dim: int   = 100,
        num_filters:   int   = 128,
        kernel_sizes:  tuple = (3, 4, 5),
        dropout_rate:  float = 0.4,
        pretrained_weights: Optional[np.ndarray] = None,
        freeze_embeddings:  bool = False,
    ):
        super().__init__()

        if pretrained_weights is not None:
            weights = torch.tensor(pretrained_weights, dtype=torch.float32)
            self.embedding = nn.Embedding.from_pretrained(
                weights, freeze=freeze_embeddings, padding_idx=0
            )
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        self.convs = nn.ModuleList([
            nn.Conv2d(1, num_filters, (k, embedding_dim))
            for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout_rate)
        self.fc      = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def _conv_pool(self, conv, x):
        # x: (batch, 1, seq, emb_dim)
        out = torch.relu(conv(x)).squeeze(3)        # (batch, filters, seq-k+1)
        return torch.max(out, dim=2).values          # (batch, filters)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb    = self.embedding(x).unsqueeze(1)     # (batch, 1, seq, emb_dim)
        pooled = [self._conv_pool(c, emb) for c in self.convs]
        cat    = self.dropout(torch.cat(pooled, dim=1))
        return self.fc(cat)


cnn = CNN1DClassifier(
    vocab_size=VOCAB_SIZE, num_classes=NUM_CLASSES,
    embedding_dim=EMBEDDING_DIM, pretrained_weights=w2v_matrix,
).to(DEVICE)
print(f"\n--- Test CNN-1D ---")
print(f"  Logits shape: {cnn(dummy_x).shape}  (esperado: [4, {NUM_CLASSES}])")
print(f"  Parámetros  : {sum(p.numel() for p in cnn.parameters()):,}")

## Modelo 4 — TF-IDF + XGBoost (Baseline clásico)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import xgboost as xgb

class TFIDFXGBoostClassifier:
    """
    Pipeline clásico con interfaz sklearn (fit / predict / predict_proba).
    sublinear_tf=True aplica log(1+tf), que amortigua el efecto de términos
    muy frecuentes sin eliminarlos completamente como haría un umbral duro.
    ngram_range=(1,2) incorpora bigramas para capturar colocaciones.
    """
    def __init__(
        self,
        max_features: int   = 30_000,
        ngram_range:  tuple = (1, 2),
        xgb_params:   dict  = None,
    ):
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=ngram_range,
            sublinear_tf=True,
            strip_accents="unicode",
            token_pattern=r"\b[a-zA-Z]{2,}\b",
        )
        default_xgb = dict(
            n_estimators=500, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="mlogloss", tree_method="hist",
            random_state=42, n_jobs=-1,
        )
        if xgb_params:
            default_xgb.update(xgb_params)
        self.model   = xgb.XGBClassifier(**default_xgb)
        self._fitted = False

    def fit(self, texts, y, eval_texts=None, eval_y=None):
        X_tr  = self.vectorizer.fit_transform(texts)
        eval_set = None
        if eval_texts is not None:
            X_val = self.vectorizer.transform(eval_texts)
            eval_set = [(X_val, eval_y)]
        self.model.fit(X_tr, y, eval_set=eval_set, verbose=False)
        self._fitted = True
        print("INFO | TF-IDF + XGBoost entrenado.")

    def predict(self, texts):
        return self.model.predict(self.vectorizer.transform(texts))

    def predict_proba(self, texts):
        return self.model.predict_proba(self.vectorizer.transform(texts))


print("\n--- TFIDFXGBoostClassifier definido correctamente ---")

## Modelo 5 — FastText (Bag-of-Embeddings)

In [ ]:
class FastTextMeanClassifier(nn.Module):
    """
    Replica la arquitectura original de FastText: promedio de vectores de
    embedding de todos los tokens, seguido de una capa lineal de clasificación.

    La máscara de padding (tokens con índice 0) garantiza que los vectores
    nulos no arrastren el promedio — equivalente a promediar solo sobre
    los tokens reales sin necesidad de conocer la longitud de cada secuencia.
    """
    def __init__(
        self,
        vocab_size:    int,
        num_classes:   int,
        embedding_dim: int   = 100,
        dropout_rate:  float = 0.3,
        pretrained_weights: Optional[np.ndarray] = None,
        freeze_embeddings:  bool = False,
    ):
        super().__init__()

        if pretrained_weights is not None:
            weights = torch.tensor(pretrained_weights, dtype=torch.float32)
            self.embedding = nn.Embedding.from_pretrained(
                weights, freeze=freeze_embeddings, padding_idx=0
            )
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        self.dropout = nn.Dropout(dropout_rate)
        self.fc      = nn.Linear(embedding_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len)
        mask     = (x != 0).float().unsqueeze(2)           # (batch, seq, 1)
        emb      = self.embedding(x) * mask                # (batch, seq, emb_dim)
        lengths  = mask.sum(dim=1).clamp(min=1)            # (batch, 1)
        mean_emb = emb.sum(dim=1) / lengths                # (batch, emb_dim)
        return self.fc(self.dropout(mean_emb))


ft_matrix = np.load("artifacts/ft_matrix.npy")
ft_clf = FastTextMeanClassifier(
    vocab_size=VOCAB_SIZE, num_classes=NUM_CLASSES,
    embedding_dim=EMBEDDING_DIM, pretrained_weights=ft_matrix,
).to(DEVICE)
print(f"\n--- Test FastText Mean ---")
print(f"  Logits shape: {ft_clf(dummy_x).shape}  (esperado: [4, {NUM_CLASSES}])")
print(f"  Parámetros  : {sum(p.numel() for p in ft_clf.parameters()):,}")

## Resumen de arquitecturas

In [ ]:
import pandas as pd

resumen = pd.DataFrame([
    {"Modelo": "DistilBERT",        "Tipo": "Transformer (fine-tune)", "Embeddings": "Contextuales"},
    {"Modelo": "BiLSTM",            "Tipo": "RNN bidireccional",       "Embeddings": "Word2Vec"},
    {"Modelo": "CNN-1D multi-kernel","Tipo": "Conv. temporal",          "Embeddings": "Word2Vec"},
    {"Modelo": "TF-IDF + XGBoost",  "Tipo": "Clásico (GBT)",           "Embeddings": "TF-IDF"},
    {"Modelo": "FastText Mean",      "Tipo": "Bag-of-Embeddings",       "Embeddings": "FastText"},
])
print(resumen.to_string(index=False))

# Flag de validación para que el Notebook 4 pueda verificar que este se ejecutó
Path("artifacts/models_ready.flag").touch()
print("\nINFO | Todas las arquitecturas validadas. Continuar con Notebook 4.")

In [1]:
from pathlib import Path

# Creamos la bandera de validación manualmente
Path("artifacts/models_ready.flag").touch()

print("Bandera 'models_ready.flag' creada sin usar Torch")


Bandera 'models_ready.flag' creada sin usar Torch
